# LoRA sur Colab — un fine-tuning, une fois, pour savoir en parler

Objectif de la lecon 6.3.1 : transformer la theorie de l'entree glossaire 1.2.6 en **vecu chiffre** — pas devenir ML Engineer.

La bonne tache pour la demo : du **style/format**, pas des faits — apprendre au modele a repondre dans un format fixe. Prendre une tache "connaissance" ne ferait que demontrer par l'echec pourquoi le RAG existe (2.3.5).

Ce qu'on mesure et retient (a reporter dans l'entree glossaire 1.2.6) :
- temps reel d'entrainement sur T4 gratuit : **A MESURER**
- taille du dataset qui commence a marcher : **A MESURER**
- avant/apres sur un petit jeu de test : **A MESURER**
- la fragilite : oubli catastrophique si on pousse, sensibilite au template d'inference

> Runtime Colab : GPU T4 (Execution > Modifier le type d'execution).

In [ ]:
# Les briques du pipeline minimal : PEFT (LoRA) + bitsandbytes (4-bit)
%pip install -q transformers peft bitsandbytes datasets accelerate trl

In [ ]:
# Le dataset JOUET de style : ~100 exemples au format chat, tous dans
# le meme format de sortie (fiche a trois champs). C'est le cas ou le
# fine-tuning a un sens reel : un FORMAT recurrent, pas des faits.
# La qualite du dataset domine tous les hyperparametres (1.2.6).
import json, random

SUJETS = ["docker", "un NAS", "un reverse proxy", "une base vectorielle",
          "le KV cache", "un embedding", "un conteneur", "une API REST",
          "le RAID", "un VLAN", "un webhook", "le DNS"]

def exemple(sujet):
    return {"messages": [
        {"role": "user", "content": f"Explique {sujet}."},
        {"role": "assistant", "content":
         f"DEFINITION : {sujet} en une phrase.\n"
         f"ANALOGIE : une image du quotidien.\n"
         f"PIEGE : l'erreur classique du debutant."},
    ]}

donnees = [exemple(s) for s in SUJETS for _ in range(8)]
random.shuffle(donnees)
print(f"{len(donnees)} exemples — le VRAI travail d'un fine-tuning",
      "serieux serait ici (des jours de curation, 6.3.1)")

In [ ]:
# QLoRA : base quantisee 4-bit (NF4) GELEE + adaptateurs entraines en
# pleine precision — c'est ce qui fait tenir un fine-tuning sur T4.
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)

BASE = "Qwen/Qwen2.5-0.5B-Instruct"   # petit expres : la demo, pas la prod

tokenizer = AutoTokenizer.from_pretrained(BASE)
modele = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    ),
    device_map="auto",
)
print(f"base chargee : {BASE} (4-bit, gelee)")

In [ ]:
# AVANT : la reponse de la base seule sur une question de test
# (sujet ABSENT du dataset — on teste la generalisation du FORMAT).
QUESTION_TEST = "Explique un load balancer."

def generer(m, question):
    entree = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        return_tensors="pt", add_generation_prompt=True,
    ).to(m.device)
    sortie = m.generate(entree, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(sortie[0][entree.shape[1]:],
                            skip_special_tokens=True)

print("AVANT :\n", generer(modele, QUESTION_TEST))

In [ ]:
# L'entrainement LoRA : r=16, alpha=32 — ~0.5 % des parametres.
# Peu d'epochs EXPRES : sur-entrainer un petit modele = il apprend le
# dataset par coeur et perd ses capacites generales (la moitie de la
# lecon, a constater en poussant num_train_epochs a 10+).
import time
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=modele,
    train_dataset=Dataset.from_list(donnees),
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                           task_type="CAUSAL_LM"),
    args=SFTConfig(output_dir="lora-format", num_train_epochs=3,
                   per_device_train_batch_size=4, learning_rate=2e-4,
                   logging_steps=10, report_to="none"),
)
debut = time.time()
trainer.train()
print(f"\nentrainement : {time.time() - debut:.0f} s — LE chiffre a",
      "retenir pour l'entretien (avec la taille du dataset)")

In [ ]:
# APRES : meme question, base + adaptateur. Attendu : le format
# DEFINITION/ANALOGIE/PIEGE applique a un sujet jamais vu.
# Piege d'inference (1.2.6) : le template de chat DOIT etre celui de
# l'entrainement, sinon l'adaptateur semble "ne pas marcher".
print("APRES :\n", generer(trainer.model, QUESTION_TEST))

trainer.model.save_pretrained("adaptateur-format")
import os
taille = sum(os.path.getsize(os.path.join("adaptateur-format", f))
             for f in os.listdir("adaptateur-format")) / 1e6
print(f"\nadaptateur sauvegarde : {taille:.0f} Mo (la base : ~500 Mo",
      "en 4-bit — l'adaptateur se distribue separement, 1.2.6)")

## La conclusion honnete (a remonter dans l'entree glossaire 1.2.6)

| Ce qu'on mesure | Valeur (A REMPLIR apres execution) |
|---|---|
| temps d'entrainement (T4) | ... |
| taille du dataset | 96 exemples (jouet) |
| le format est-il appris ? | avant : ... / apres : ... |
| cout REEL total | dataset (le vrai travail) + iterations + evals |

Le ratio a dire en entretien : cote GPU c'est rapide et gratuit ; le cout reel est le **dataset propre** et les **evals** — a comparer a "ecrire un bon prompt systeme" (minutes). C'est ce ratio qui justifie l'ordre **prompt -> few-shot -> RAG -> fine-tuning** (2.3.5).

Bonus a mentionner (pas forcement a faire) : vLLM (4.1.1) sait charger des adaptateurs LoRA — l'adaptateur pourrait se servir sur le homelab.